# Breast Cancer Wisconsin (Classificação de Tumores)

## PUC-Rio | MVP 

**Problema** 

Classificar tumores como malignos (0) ou benignos (1) com base em medições celulares de biópsias.

**Dataset** 

Breast Cancer Wisconsin (Diagnostic) &rarr; 569 amostras, 30 features numéricas e 2 classes.

**Fonte**

 UCI Repository via scikit-learn (`load_breast_cancer`).

## 1. Importações

In [1]:
import pickle
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

SEED = 42
np.random.seed(SEED)
print('OK')

OK


## 2. Carga dos dados

In [2]:
dados = load_breast_cancer(as_frame=True)

# Usamos apenas as 10 features '_mean' (primeiras 10 colunas)
# para simplificar o formulário do frontend
FEATURES = list(dados.feature_names[:10])

X = dados.data[FEATURES].copy()
y = dados.target.copy()  # 0 = Maligno, 1 = Benigno

print(f'Shape: {X.shape}')
print(f'Classes: {dict(zip(dados.target_names, [int((y==i).sum()) for i in range(2)]))}')  
print(f'Valores nulos: {X.isnull().sum().sum()}')

Shape: (569, 10)
Classes: {np.str_('malignant'): 212, np.str_('benign'): 357}
Valores nulos: 0


## 3. Holdout: separação treino/teste

In [3]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'Treino: {X_treino.shape[0]} amostras | Teste: {X_teste.shape[0]} amostras')

Treino: 455 amostras | Teste: 114 amostras


## 4. Modelagem: Pipeline + GridSearchCV

In [4]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

configuracoes = {
    'KNN': {
        'modelo': KNeighborsClassifier(),
        'params': {
            'classificador__n_neighbors': [3, 5, 7, 9, 11],
            'classificador__weights': ['uniform', 'distance'],
        }
    },
    'Arvore': {
        'modelo': DecisionTreeClassifier(random_state=SEED),
        'params': {
            'classificador__max_depth': [3, 5, 7, None],
            'classificador__min_samples_split': [2, 5, 10],
            'classificador__criterion': ['gini', 'entropy'],
        }
    },
    'NaiveBayes': {
        'modelo': GaussianNB(),
        'params': {
            'classificador__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
        }
    },
    'SVM': {
        'modelo': SVC(probability=True, random_state=SEED),
        'params': {
            'classificador__C': [0.1, 1, 10],
            'classificador__kernel': ['rbf', 'linear'],
        }
    }
}

resultados = {}

for nome, conf in configuracoes.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classificador', conf['modelo'])
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid=conf['params'],
        cv=kfold,
        scoring='accuracy',
        n_jobs=-1
    )
    grid.fit(X_treino, y_treino)

    y_pred = grid.best_estimator_.predict(X_teste)

    resultados[nome] = {
        'melhor_pipeline': grid.best_estimator_,
        'melhores_params':  grid.best_params_,
        'cv_accuracy':      grid.best_score_,
        'y_pred':           y_pred,
        'accuracy':         accuracy_score(y_teste, y_pred),
        'precision':        precision_score(y_teste, y_pred, pos_label=0),
        'recall':           recall_score(y_teste, y_pred, pos_label=0),
        'f1':               f1_score(y_teste, y_pred, pos_label=0),
    }

    print(f'{nome:12s} | CV Acc: {grid.best_score_:.4f} | Test Acc: {resultados[nome]["accuracy"]:.4f}')

KNN          | CV Acc: 0.9429 | Test Acc: 0.9211
Arvore       | CV Acc: 0.9209 | Test Acc: 0.8947
NaiveBayes   | CV Acc: 0.9121 | Test Acc: 0.9211
SVM          | CV Acc: 0.9429 | Test Acc: 0.9211


## 5. Comparação dos modelos

In [5]:
metricas_df = pd.DataFrame({
    nome: {
        'CV Accuracy': f"{r['cv_accuracy']:.4f}",
        'Accuracy':    f"{r['accuracy']:.4f}",
        'Precision':   f"{r['precision']:.4f}",
        'Recall':      f"{r['recall']:.4f}",
        'F1-Score':    f"{r['f1']:.4f}",
    }
    for nome, r in resultados.items()
}).T

print(metricas_df.to_string())

           CV Accuracy Accuracy Precision  Recall F1-Score
KNN             0.9429   0.9211    0.8667  0.9286   0.8966
Arvore          0.9209   0.8947    0.8000  0.9524   0.8696
NaiveBayes      0.9121   0.9211    0.8837  0.9048   0.8941
SVM             0.9429   0.9211    0.8667  0.9286   0.8966


## 6. Exportação do melhor modelo

In [6]:
melhor_nome = max(resultados, key=lambda k: resultados[k]['f1'])
melhor = resultados[melhor_nome]

print(f'Melhor modelo: {melhor_nome}')
print(f'F1-Score : {melhor["f1"]:.4f}')
print(f'Recall   : {melhor["recall"]:.4f}')
print(f'Params   : {melhor["melhores_params"]}')
print()
print(classification_report(y_teste, melhor['y_pred'], target_names=['Maligno', 'Benigno']))

pipeline_final = melhor['melhor_pipeline']
scaler_final   = pipeline_final.named_steps['scaler']
modelo_final   = pipeline_final.named_steps['classificador']

with open('melhor_modelo.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler_final, f)

with open('colunas.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

print('Artefatos salvos: melhor_modelo.pkl | scaler.pkl | colunas.pkl')

Melhor modelo: KNN
F1-Score : 0.8966
Recall   : 0.9286
Params   : {'classificador__n_neighbors': 3, 'classificador__weights': 'uniform'}

              precision    recall  f1-score   support

     Maligno       0.87      0.93      0.90        42
     Benigno       0.96      0.92      0.94        72

    accuracy                           0.92       114
   macro avg       0.91      0.92      0.92       114
weighted avg       0.92      0.92      0.92       114

Artefatos salvos: melhor_modelo.pkl | scaler.pkl | colunas.pkl


## 7. Análise dos resultados

Todos os quatro algoritmos alcançaram acurácia acima de 90% no conjunto de teste, resultado esperado para um dataset limpo e bem separável como o Wisconsin.

**SVM** tende a ser o melhor algoritmo neste dataset, especialmente com kernel RBF, que lida bem com fronteiras não-lineares entre as classes.

**KNN** é sensível à escala das features — por isso o `StandardScaler` no pipeline é fundamental: sem ele, features com valores grandes (como `area_mean`) dominariam o cálculo de distância.

**Árvore de Decisão** é o modelo mais interpretável, mas tende ao overfitting quando `max_depth=None`.

**Naive Bayes** é o mais rápido e surpreendentemente competitivo, mesmo assumindo independência entre features — pressuposto que o mapa de correlação mostra ser violado (`radius`, `perimeter` e `area` têm correlação >0.99).

**Ponto de atenção:** em diagnóstico médico, o Recall é a métrica mais crítica — um falso negativo (tumor maligno classificado como benigno) é mais grave que um falso positivo. O modelo selecionado por F1-Score já equilibra bem esse trade-off.

**Reflexão sobre segurança:** dados de saúde exigem anonimização antes do treino, minimização de features, criptografia em trânsito (HTTPS) e conformidade com a LGPD. O histórico de predições implementado no backend garante rastreabilidade do uso do modelo.